In [1]:
DATA = "bookings.jsonl"

# Task 2: Contracts, nested data and a star schema

Every count below is computed from `DATA`; nothing is typed in by hand.

In [2]:
import json
import math
import re
import shutil
from collections import Counter
from datetime import date, datetime
from pathlib import Path
from typing import List, Literal, Optional

import duckdb
import pandas as pd
from pydantic import BaseModel, Field, ValidationError, field_validator, model_validator

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

## Part A: The contract

Four Pydantic models: `Ticket`, `Student`, `Event` and `Booking`, which uses the other three.
Single-field rules use types (`Literal`, `Field(ge=...)`, a regex). The two cross-field rules
(total = Σ qty × price, and booked before the event date) are in a `model_validator`.

In [3]:
Faculty = Literal["MET", "IET", "EMS", "Pharmacy", "Management"]
Club = Literal["Robotics", "Debate", "Music", "Football", "Chess"]
TicketType = Literal["standard", "vip", "student"]
Payment = Literal["card", "cash", "wallet"]
Status = Literal["confirmed", "cancelled"]

STUDENT_ID = re.compile(r"\d{2}-\d{4}")


class Ticket(BaseModel):
    type: TicketType
    qty: int = Field(ge=1)
    price: float = Field(ge=0)


class Student(BaseModel):
    id: str
    name: str
    faculty: Faculty

    @field_validator("id")
    @classmethod
    def id_format(cls, v: str) -> str:
        if not STUDENT_ID.fullmatch(v):
            raise ValueError("student id must be two digits, a dash, four digits (e.g. 58-1234)")
        return v


class Event(BaseModel):
    id: str
    title: str
    club: Club
    date: date
    venue: str


class Booking(BaseModel):
    booking_id: str
    booked_at: datetime
    student: Student
    event: Event                      # required: a missing / null event fails
    tickets: List[Ticket] = Field(min_length=1)
    total: float
    payment: Payment
    status: Status
    promo_code: Optional[str] = None  # only in later bookings (schema drift)

    @model_validator(mode="after")
    def cross_field_rules(self):
        expected = sum(t.qty * t.price for t in self.tickets)
        if not math.isclose(self.total, expected, abs_tol=0.005):
            raise ValueError("total does not equal sum of qty x price")
        if self.booked_at.date() >= self.event.date:
            raise ValueError("booked_at is not before the event date")
        return self

### Validate every line, then remove re-sent bookings

Each line is parsed and validated. A failing line goes to the quarantine with its `booking_id`
and a reason. The reason is built from Pydantic's error: the field path (with list positions
dropped, so `tickets.0.qty` and `tickets.1.qty` count as the same reason) plus the message.

Only **after** validation are duplicates removed: the first valid copy of each `booking_id`
is kept and later valid copies are quarantined as `duplicate booking_id`.

In [4]:
def reason_from(err: ValidationError) -> str:
    """One readable reason per booking: field path (no list indexes) + message."""
    parts = []
    for e in err.errors():
        loc = ".".join(str(p) for p in e["loc"] if not isinstance(p, int))
        msg = e["msg"].removeprefix("Value error, ")
        parts.append(f"{loc}: {msg}" if loc else msg)
    return "; ".join(dict.fromkeys(parts))   # de-duplicate, keep order


lines_read = 0
valid = []          # (line_no, raw dict, Booking) for bookings that pass the contract
quarantine = []     # {"line": .., "booking_id": .., "reason": ..}

with open(DATA, encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue
        lines_read += 1
        try:
            raw = json.loads(line)
        except json.JSONDecodeError:
            quarantine.append({"line": line_no, "booking_id": None, "reason": "invalid JSON"})
            continue
        if not isinstance(raw, dict):
            quarantine.append({"line": line_no, "booking_id": None, "reason": "not a JSON object"})
            continue
        try:
            booking = Booking.model_validate(raw)
        except ValidationError as err:
            quarantine.append({"line": line_no, "booking_id": raw.get("booking_id"),
                               "reason": reason_from(err)})
            continue
        valid.append((line_no, raw, booking))

# Re-sent bookings: keep the first valid copy of each booking_id
seen = set()
kept = []
for line_no, raw, booking in valid:
    if booking.booking_id in seen:
        quarantine.append({"line": line_no, "booking_id": booking.booking_id,
                           "reason": "duplicate booking_id"})
    else:
        seen.add(booking.booking_id)
        kept.append((line_no, raw, booking))

quarantine.sort(key=lambda q: q["line"])
with open("quarantine.jsonl", "w", encoding="utf-8") as f:
    for q in quarantine:
        f.write(json.dumps(q) + "\n")

print(f"quarantine.jsonl written: {len(quarantine)} lines")

quarantine.jsonl written: 24 lines


### Report

In [5]:
reason_counts = Counter(q["reason"] for q in quarantine)

print(f"Lines read:          {lines_read}")
print(f"Passed (kept):       {len(kept)}")
print(f"Quarantined:         {len(quarantine)}")
assert lines_read == len(kept) + len(quarantine)

report = (pd.DataFrame(reason_counts.items(), columns=["reason", "count"])
            .sort_values(["count", "reason"], ascending=[False, True])
            .reset_index(drop=True))
report

Lines read:          283
Passed (kept):       259
Quarantined:         24


,reason,count
0,"student.id: student id must be two digits, a d...",4
1,total does not equal sum of qty x price,4
2,booked_at is not before the event date,3
3,duplicate booking_id,3
4,tickets.qty: Input should be greater than or e...,3
5,"tickets.type: Input should be 'standard', 'vip...",3
6,event: Input should be a valid dictionary or i...,2
7,tickets: List should have at least 1 item afte...,2


In [6]:
# The quarantined bookings themselves
pd.DataFrame(quarantine)

,line,booking_id,reason
0,11,B0011,"tickets.type: Input should be 'standard', 'vip..."
1,16,B0016,tickets.qty: Input should be greater than or e...
2,24,B0024,"student.id: student id must be two digits, a d..."
3,44,B0035,duplicate booking_id
4,50,B0049,event: Input should be a valid dictionary or i...
5,58,B0057,"student.id: student id must be two digits, a d..."
6,61,B0060,"student.id: student id must be two digits, a d..."
7,91,B0090,"tickets.type: Input should be 'standard', 'vip..."
8,102,B0101,total does not equal sum of qty x price
9,107,B0106,"student.id: student id must be two digits, a d..."


#### Would the counts change if duplicates were removed *before* validation?

Same rules, other order, for comparison only (nothing below is used later). If a re-sent
booking is invalid, dedup-first labels its second copy `duplicate booking_id` instead of its real
problem; and if the *first* copy is invalid but a later copy is valid, dedup-first throws away the
only good copy.

In [7]:
def outcome(raw):
    try:
        Booking.model_validate(raw)
        return None
    except ValidationError as err:
        return reason_from(err)

swapped_reasons, swapped_seen, swapped_kept = Counter(), set(), 0
with open(DATA, encoding="utf-8") as f:
    lines = [line for line in f if line.strip()]
for line in lines:
    try:
        r = json.loads(line)
    except json.JSONDecodeError:
        swapped_reasons["invalid JSON"] += 1
        continue
    if not isinstance(r, dict):
        swapped_reasons["not a JSON object"] += 1
        continue
    bid = r.get("booking_id")
    if bid in swapped_seen:
        swapped_reasons["duplicate booking_id"] += 1
        continue
    swapped_seen.add(bid)
    reason = outcome(r)
    if reason:
        swapped_reasons[reason] += 1
    else:
        swapped_kept += 1

print(f"validate, then dedup:    kept {len(kept)}, duplicates {reason_counts['duplicate booking_id']}")
print(f"dedup, then validate:    kept {swapped_kept}, duplicates {swapped_reasons['duplicate booking_id']}")
diff = {k: (reason_counts[k], swapped_reasons[k])
        for k in reason_counts | swapped_reasons if reason_counts[k] != swapped_reasons[k]}
print("reasons whose count differs (used order, swapped order):", diff or "none, same counts on this file")

validate, then dedup:    kept 259, duplicates 3
dedup, then validate:    kept 259, duplicates 3
reasons whose count differs (used order, swapped order): none, same counts on this file


## Part B: Flatten

`bookings`: one row per booking, student and event objects become `student_*` / `event_*` columns.
`booking_tickets`: one row per ticket entry, with `amount = qty × price`.

Values come from the validated `Booking` objects (so `booked_at` is a real timestamp and `event.date`
a real date). The raw document is only used to see whether `promo_code` was actually sent.

In [8]:
records = []
for _, raw, b in kept:
    d = b.model_dump()
    d["has_promo_code"] = "promo_code" in raw   # was the field in the document at all?
    records.append(d)

bookings = pd.json_normalize(records, sep="_").drop(columns=["tickets"])
bookings["booked_at"] = pd.to_datetime(bookings["booked_at"], utc=True)
bookings["event_date"] = pd.to_datetime(bookings["event_date"]).dt.date
bookings = bookings[["booking_id", "booked_at",
                     "student_id", "student_name", "student_faculty",
                     "event_id", "event_title", "event_club", "event_date", "event_venue",
                     "total", "payment", "status", "promo_code", "has_promo_code"]]

booking_tickets = pd.json_normalize(records, record_path="tickets", meta=["booking_id"])
booking_tickets = booking_tickets[["booking_id", "type", "qty", "price"]]
booking_tickets["amount"] = booking_tickets["qty"] * booking_tickets["price"]

print("bookings:", bookings.shape, "  booking_tickets:", booking_tickets.shape)
bookings.head()

bookings: (259, 15)   booking_tickets: (314, 5)


,booking_id,booked_at,student_id,student_name,student_faculty,event_id,event_title,event_club,event_date,event_venue,total,payment,status,promo_code,has_promo_code
0,B0001,2026-09-28 15:46:00+00:00,60-3778,Nour Mansour,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,340.0,card,cancelled,NaN,False
1,B0002,2026-09-27 12:29:00+00:00,61-3826,Laila Adel,IET,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A,40.0,card,confirmed,NaN,False
2,B0003,2026-10-13 20:19:00+00:00,58-5324,Ziad Kamel,IET,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium,60.0,wallet,confirmed,NaN,False
3,B0004,2026-11-01 06:42:00+00:00,58-6052,Seif Ibrahim,Management,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage,105.0,card,confirmed,CLUB10,True
4,B0005,2026-10-28 19:06:00+00:00,58-9891,Ali Fouad,Management,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A,20.0,cash,confirmed,NaN,True


In [9]:
booking_tickets.head()

,booking_id,type,qty,price,amount
0,B0001,standard,1,40.0,40.0
1,B0001,vip,3,100.0,300.0
2,B0002,standard,1,40.0,40.0
3,B0003,standard,2,30.0,60.0
4,B0004,standard,3,35.0,105.0


### Checks

In [10]:
# 1. booking_id is unique in bookings
assert bookings["booking_id"].is_unique, "booking_id is not unique"

# 2. for every booking, the amounts of its tickets add up to its total
sums = booking_tickets.groupby("booking_id")["amount"].sum()
check = bookings.set_index("booking_id")["total"].to_frame().join(sums)
assert check["amount"].notna().all(), "a booking has no tickets"
assert ((check["total"] - check["amount"]).abs() < 0.005).all(), "ticket amounts != total"

# every ticket belongs to a booking
assert booking_tickets["booking_id"].isin(bookings["booking_id"]).all()
print("All Part B assertions passed.")

All Part B assertions passed.


### Schema drift

Compare each document's keys (top level and inside `student`, `event`, each ticket) with the
example document in the brief, then find the first booking (by `booked_at`) that has the extra field.

In [11]:
EXPECTED = {
    "": {"booking_id", "booked_at", "student", "event", "tickets", "total", "payment", "status"},
    "student": {"id", "name", "faculty"},
    "event": {"id", "title", "club", "date", "venue"},
    "tickets[]": {"type", "qty", "price"},
}

def extra_keys(raw):
    found = set(raw) - EXPECTED[""]
    if isinstance(raw.get("student"), dict):
        found |= {f"student.{k}" for k in set(raw["student"]) - EXPECTED["student"]}
    if isinstance(raw.get("event"), dict):
        found |= {f"event.{k}" for k in set(raw["event"]) - EXPECTED["event"]}
    for t in raw.get("tickets") or []:
        if isinstance(t, dict):
            found |= {f"tickets[].{k}" for k in set(t) - EXPECTED["tickets[]"]}
    return found

drift = Counter(k for _, raw, _ in kept for k in extra_keys(raw))
print("Extra fields in valid bookings:", dict(drift))

# bookings rows are in the same order as kept, so the flags line up row by row
for field in drift:
    present = pd.Series([field in extra_keys(raw) for _, raw, _ in kept], index=bookings.index)
    first = bookings.loc[present, "booked_at"].min()
    from_then = bookings["booked_at"] >= first
    print(f"\n'{field}':")
    print(f"  present in {present.sum()} of {len(bookings)} valid bookings")
    print(f"  first appears in a booking made at {first:%Y-%m-%d %H:%M} UTC")
    print(f"  bookings before that with the field: {(present & ~from_then).sum()} of {(~from_then).sum()}")
    print(f"  bookings from then on without it:    {(~present & from_then).sum()} of {from_then.sum()}")

Extra fields in valid bookings: {'promo_code': 173}

'promo_code':
  present in 173 of 259 valid bookings
  first appears in a booking made at 2026-10-20 01:05 UTC
  bookings before that with the field: 0 of 86
  bookings from then on without it:    0 of 173


In [12]:
# What the field holds, before and after it appears
print(bookings.groupby("has_promo_code")["booked_at"].agg(["count", "min", "max"]))
print()
print(bookings.loc[bookings["has_promo_code"], "promo_code"].value_counts(dropna=False))

                count                       min                       max
has_promo_code                                                           
False              86 2026-09-25 12:57:00+00:00 2026-10-19 18:43:00+00:00
True              173 2026-10-20 01:05:00+00:00 2026-11-24 17:09:00+00:00

promo_code
NaN       86
CLUB10    47
EARLY     40
Name: count, dtype: int64


Documents without `promo_code` are still valid (the field is `Optional` with default `None`),
so they get a null `promo_code` in `bookings`. `has_promo_code` keeps the difference between
"field not sent" (old app version) and "sent as null" (new version, no code used).

## Part C: Star schema in DuckDB

* `dim_event`: one row per event, surrogate key `event_key`
* `dim_student`: one row per student, surrogate key `student_key`
* `fact_ticket_sales`: one row per ticket entry, with `date_key` = `booked_at` as `YYYYMMDD` (UTC)

In [13]:
con = duckdb.connect()

def run_sql(sql):
    """Run a SQL statement and return all result rows."""
    return con.execute(sql).fetchall()

run_sql("SET TimeZone = 'UTC'")

# DuckDB reads the pandas frames; booked_at as a plain UTC timestamp
bookings_db = bookings.assign(booked_at=bookings["booked_at"].dt.tz_convert(None))
con.register("bookings", bookings_db)
con.register("booking_tickets", booking_tickets)

run_sql("""
CREATE OR REPLACE TABLE dim_event AS
SELECT ROW_NUMBER() OVER (ORDER BY event_id) AS event_key, *
FROM (
    SELECT event_id,
           arg_min(event_title, booked_at) AS title,
           arg_min(event_club,  booked_at) AS club,
           arg_min(event_date,  booked_at) AS event_date,
           arg_min(event_venue, booked_at) AS venue
    FROM bookings
    GROUP BY event_id
)
""")

run_sql("""
CREATE OR REPLACE TABLE dim_student AS
SELECT ROW_NUMBER() OVER (ORDER BY student_id) AS student_key, *
FROM (
    SELECT student_id,
           arg_min(student_name,    booked_at) AS name,
           arg_min(student_faculty, booked_at) AS faculty
    FROM bookings
    GROUP BY student_id
)
""")

run_sql("""
CREATE OR REPLACE TABLE fact_ticket_sales AS
SELECT t.booking_id,
       e.event_key,
       s.student_key,
       t.type AS ticket_type,
       t.qty,
       t.price,
       t.amount,
       b.status,
       CAST(strftime(b.booked_at, '%Y%m%d') AS INTEGER) AS date_key,
       strftime(b.booked_at, '%Y-%m')                   AS booked_month
FROM booking_tickets t
JOIN bookings    b ON b.booking_id = t.booking_id
JOIN dim_event   e ON e.event_id   = b.event_id
JOIN dim_student s ON s.student_id = b.student_id
""")

for t in ["dim_event", "dim_student", "fact_ticket_sales"]:
    print(t, run_sql(f"SELECT COUNT(*) FROM {t}")[0][0], "rows")

dim_event 8 rows


dim_student 103 rows
fact_ticket_sales 314 rows


In [14]:
con.sql("SELECT * FROM dim_event").df()

,event_key,event_id,title,club,event_date,venue
0,1,EV-01,Robotics Demo Day,Robotics,2026-10-15,Hall A
1,2,EV-02,Inter-faculty Debate Final,Debate,2026-10-22,Auditorium
2,3,EV-03,Autumn Concert,Music,2026-10-29,Open-air theatre
3,4,EV-04,Five-a-side Cup,Football,2026-11-05,Sports field
4,5,EV-05,Blitz Chess Open,Chess,2026-11-07,C7.201
5,6,EV-06,Line-follower Challenge,Robotics,2026-11-12,Hall A
6,7,EV-07,Open Mic Night,Music,2026-11-19,Cafeteria stage
7,8,EV-08,Model UN Debate,Debate,2026-11-26,Auditorium


In [15]:
con.sql("SELECT * FROM dim_student LIMIT 5").df()

,student_key,student_id,name,faculty
0,1,58-1324,Nour Fouad,MET
1,2,58-1449,Farida Mansour,EMS
2,3,58-1801,Karim Kamel,IET
3,4,58-2165,Farida Hassan,Management
4,5,58-2307,Farida Hassan,Pharmacy


In [16]:
con.sql("SELECT * FROM fact_ticket_sales LIMIT 5").df()

,booking_id,event_key,student_key,ticket_type,qty,price,amount,status,date_key,booked_month
0,B0001,1,61,standard,1,40.0,40.0,cancelled,20260928,2026-09
1,B0001,1,61,vip,3,100.0,300.0,cancelled,20260928,2026-09
2,B0002,1,84,standard,1,40.0,40.0,confirmed,20260927,2026-09
3,B0003,2,14,standard,2,30.0,60.0,confirmed,20261013,2026-10
4,B0004,7,15,standard,3,35.0,105.0,confirmed,20261101,2026-11


### Checks

In [17]:
fact_rows = run_sql("SELECT COUNT(*) FROM fact_ticket_sales")[0][0]
fact_amount = run_sql("SELECT SUM(amount) FROM fact_ticket_sales")[0][0]
bookings_total = run_sql("SELECT SUM(total) FROM bookings")[0][0]

assert fact_rows == len(booking_tickets), f"{fact_rows} fact rows vs {len(booking_tickets)} tickets"
assert math.isclose(fact_amount, bookings_total, abs_tol=0.005), f"{fact_amount} vs {bookings_total}"

# surrogate keys are unique, and every natural key appears once
for dim, key, nat in [("dim_event", "event_key", "event_id"), ("dim_student", "student_key", "student_id")]:
    n, k, i = run_sql(f"SELECT COUNT(*), COUNT(DISTINCT {key}), COUNT(DISTINCT {nat}) FROM {dim}")[0]
    assert n == k == i, dim

print(f"fact rows = booking_tickets rows = {fact_rows}")
print(f"SUM(amount) in fact = SUM(total) in bookings = {fact_amount:,.2f}")

fact rows = booking_tickets rows = 314
SUM(amount) in fact = SUM(total) in bookings = 31,285.00


### Confirmed bookings: tickets sold and revenue per club and ticket type

In [18]:
answer = con.sql("""
SELECT e.club,
       f.ticket_type,
       CAST(SUM(f.qty) AS BIGINT) AS tickets_sold,
       SUM(f.amount) AS revenue_egp
FROM fact_ticket_sales f
JOIN dim_event e USING (event_key)
WHERE f.status = 'confirmed'
GROUP BY e.club, f.ticket_type
ORDER BY e.club, f.ticket_type
""").df()
answer

,club,ticket_type,tickets_sold,revenue_egp
0,Chess,standard,45,900.0
1,Chess,student,44,440.0
2,Debate,standard,65,1950.0
3,Debate,student,34,510.0
4,Debate,vip,40,3000.0
5,Football,standard,64,1600.0
6,Music,standard,67,2930.0
7,Music,student,20,1000.0
8,Music,vip,78,9990.0
9,Robotics,standard,60,2400.0


### Save as Parquet in `star/`

In [19]:
STAR = Path("star")
if STAR.exists():
    shutil.rmtree(STAR)          # start clean so a re-run leaves no stale files
STAR.mkdir()

run_sql("COPY dim_event   TO 'star/dim_event.parquet'   (FORMAT PARQUET)")
run_sql("COPY dim_student TO 'star/dim_student.parquet' (FORMAT PARQUET)")
run_sql("""
COPY fact_ticket_sales TO 'star/fact_ticket_sales'
     (FORMAT PARQUET, PARTITION_BY (booked_month))
""")

for p in sorted(STAR.rglob("*.parquet")):
    print(p.as_posix())

star/dim_event.parquet
star/dim_student.parquet
star/fact_ticket_sales/booked_month=2026-09/data_0.parquet
star/fact_ticket_sales/booked_month=2026-10/data_0.parquet
star/fact_ticket_sales/booked_month=2026-11/data_0.parquet


In [20]:
# Read the star back from Parquet and check it matches
back = run_sql("""
SELECT COUNT(*), SUM(amount)
FROM read_parquet('star/fact_ticket_sales/*/*.parquet', hive_partitioning = true)
""")[0]
assert back[0] == fact_rows and math.isclose(back[1], fact_amount, abs_tol=0.005)
print("Parquet fact:", back[0], "rows, SUM(amount) =", back[1])

Parquet fact: 314 rows, SUM(amount) = 31285.0


### Summary

In [21]:
promo_first = bookings.loc[bookings["has_promo_code"], "booked_at"].min()
total_over_fact = run_sql("""
    SELECT SUM(b.total) FROM fact_ticket_sales f JOIN bookings b USING (booking_id)
""")[0][0]

print(f"Lines read: {lines_read}")
for reason, n in report.itertuples(index=False):
    print(f"  quarantined - {reason}: {n}")
print(f"Total quarantined: {len(quarantine)}")
print(f"Bookings left: {len(bookings)}  (ticket entries: {len(booking_tickets)})")
print(f"promo_code first appears: {promo_first}")
print(f"SUM(total) repeated on every fact row: {total_over_fact:,.2f}"
      f"  vs real revenue {bookings_total:,.2f}")

Lines read: 283
  quarantined - student.id: student id must be two digits, a dash, four digits (e.g. 58-1234): 4
  quarantined - total does not equal sum of qty x price: 4
  quarantined - booked_at is not before the event date: 3
  quarantined - duplicate booking_id: 3
  quarantined - tickets.qty: Input should be greater than or equal to 1: 3
  quarantined - tickets.type: Input should be 'standard', 'vip' or 'student': 3
  quarantined - event: Input should be a valid dictionary or instance of Event: 2
  quarantined - tickets: List should have at least 1 item after validation, not 0: 2
Total quarantined: 24
Bookings left: 259  (ticket entries: 314)
promo_code first appears: 2026-10-20 01:05:00+00:00
SUM(total) repeated on every fact row: 42,000.00  vs real revenue 31,285.00
